In [1]:
import os

In [2]:
!pwd

/Users/abhikchoudhury/Library/CloudStorage/OneDrive-IBM/Badlo/Krish Naik Projects/Kidney_Tumor_Identification_System_Computer_vision/Kidney Tumor Identification System_Abhik/research


In [3]:
os.chdir("../")

In [4]:
pwd

'/Users/abhikchoudhury/Library/CloudStorage/OneDrive-IBM/Badlo/Krish Naik Projects/Kidney_Tumor_Identification_System_Computer_vision/Kidney Tumor Identification System_Abhik'

This module implements a transfer learning pipeline using a pretrained VGG16 architecture. It loads the ImageNet weights, freezes the convolutional base to preserve learned feature representations, and adds a custom classification head using a dense softmax layer. The model is then compiled with SGD optimizer and categorical cross-entropy loss, enabling it to be fine-tuned for a domain-specific multi-class classification task. The design is config-driven and modular, making it production-ready and reusable

In [ ]:
from dataclasses import dataclass
from pathlib import Path
#here again, just like the data ingestion notebook, we are using an entity class to fix the return types. 


@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir: Path # defined in config.yaml
    base_model_path: Path # defined in config.yaml
    updated_base_model_path: Path # defined in config.yaml
    params_image_size: list # defined in params.yaml
    params_learning_rate: float # defined in params.yaml
    params_include_top: bool # defined in params.yaml
    params_weights: str # defined in params.yaml

    params_classes: int # defined in params.yaml

 #once this notebook is run, move this to cnnClassifier/entity/config_entity.py   

In [ ]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

#JUST LIKE THE 01_DATA_INGESTION NOTEBOOK, we are importing the params.yaml and config.yaml path from the cnnclassifier/constants folder.
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])


    def get_prepare_base_model_config(self) -> PrepareBaseModelConfig:
        config = self.config.prepare_base_model
        
        create_directories([config.root_dir])

        prepare_base_model_config = PrepareBaseModelConfig(
            root_dir=Path(config.root_dir),
            base_model_path=Path(config.base_model_path),
            updated_base_model_path=Path(config.updated_base_model_path),
            params_image_size=self.params.IMAGE_SIZE,
            params_learning_rate=self.params.LEARNING_RATE,
            params_include_top=self.params.INCLUDE_TOP,
            params_weights=self.params.WEIGHTS,
            params_classes=self.params.CLASSES
        )

        return prepare_base_model_config

 #once this notebook is run, move this (get_prepare_base_model_config(self)) to cnnClassifier/configuration.py

In [9]:
#Now we need to upgrade my components. First download the libraries
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf

In [ ]:
#In this class, we prepare the base model, which will accept the preparebasemodel config defined earlier. 
class PrepareBaseModel:
    def __init__(self, config: PrepareBaseModelConfig):
        self.config = config

#download the vgg16 model from the Keras. How to download is given here #https://keras.io/api/applications/vgg/vgg_models/#vgg16-function
    
    def get_base_model(self):
        self.model = tf.keras.applications.vgg16.VGG16(
            input_shape=self.config.params_image_size,
            weights=self.config.params_weights,
            include_top=self.config.params_include_top
        )
#also saving this model
        self.save_model(path=self.config.base_model_path, model=self.model)


    #here you are doing the modification to the base downloaded model. 
    @staticmethod
    def _prepare_full_model(model, classes, freeze_all, freeze_till, learning_rate):
        #Freeze all :Use pretrained features only, Freeze partially ine-tune last layers, No freezing Full retraining
        if freeze_all:
            #Case 1: Freeze all layers: freeze each layer individually
            for layer in model.layers:
                model.trainable = False
                #Case 2: Partial Freezing
        elif (freeze_till is not None) and (freeze_till > 0):
            for layer in model.layers[:-freeze_till]:
                model.trainable = False

#Step 2: Add Custom Layers. Converts feature maps → 1D vector. Example (7, 7, 512) → (25088,)
#Final classification layer, units=classes is Number of output categories. Since it is 2, binary classification, so Softmax was used. 
        flatten_in = tf.keras.layers.Flatten()(model.output)
        prediction = tf.keras.layers.Dense(
            units=classes,
            activation="softmax"
        )(flatten_in)
# Step 3: Creates full model by combining Original CNN base with new classification head. This is Transfer learning.  
        full_model = tf.keras.models.Model(
            inputs=model.input,
            outputs=prediction
        )

#Step 4 model compile where SGD is used. Oftern ADAM is used instead of SGD for faster convergence, SGD is more suitable for fine tuning.
# #CategoricalCrossentropy() is used for multi class classification. Metrics us accuracy.  
       full_model.compile(
            optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
            loss=tf.keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"]
        )

        full_model.summary()
        return full_model
    
#The _prepare_full_model function is responsible for constructing and compiling the transfer learning architecture, but it does not perform any training. 
# The update_base_model method acts as a pipeline step that invokes this function, stores the resulting model, and persists it to disk. 
# This separation aligns with modular ML pipeline design, where model preparation, training, and evaluation are handled as independent stages.”
#You will see later in this notebook in the pipeline that the _prepare_full_model is not called, it is actually the update_base_model which gets called and executes the _prepare_full_model
    def update_base_model(self):
        self.full_model = self._prepare_full_model(
            model=self.model,
            classes=self.config.params_classes,
            freeze_all=True,
            freeze_till=None,
            learning_rate=self.config.params_learning_rate
        )

        self.save_model(path=self.config.updated_base_model_path, model=self.full_model)

    
    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

 #once this notebook is run, move this (get_prepare_base_model_config(self)) to cnnClassifier/components/prepare_base_model.py

A point of the flatten funciton used. 
We do it to bridge CNN → Fully Connected layer. When you use a pretrained model like VGG16 (with include_top=False), the output is not a vector.

It’s a 3D tensor (feature maps).

Example:

If your input is (224, 224, 3), the output of the convolutional base is: (7, 7, 512)

7 × 7 → spatial dimensions
512 → number of feature channels (filters)

Our final layer is Dense(units=classes, activation="softmax"). Dense layers expect input like:batch_size, features.

Flatten is doing: “Take all learned spatial features and line them up into a single feature vector”

So instead of: [feature map structure] You get: [feature1, feature2, feature3, ..., feature25088].

Problem with Flatten:

Creates huge number of parameters. Example: 25,088 inputs × 10 classes = ~250K weights Can lead to: Overfitting and Slower training.

Better option is GlobalAveragePooling2D() which converts (7, 7, 512) → (512,)
Flatten is still used where You want fine-grained spatial learning. Otherwise mostly GlobalAveragePooling2D() is used.


#### Do a bit of research on Transfer learning. This is a classic use case of Transfer learning


In [ ]:
#finally we have the pipeline which calls the specific functions till model updation.  
try:
    config = ConfigurationManager()
    prepare_base_model_config = config.get_prepare_base_model_config()
    prepare_base_model = PrepareBaseModel(config=prepare_base_model_config)
    prepare_base_model.get_base_model()
    prepare_base_model.update_base_model()
except Exception as e:
    raise e
#Once this pipeline is executed, the base model and the updated models will be created in the artifacts folders.     

#Once this notebook is run, we will put this pipeline component in the cnnClassifier/pipeline/stage_02_prepare_base_model.py

[2026-05-05 00:00:03,951: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-05-05 00:00:03,955: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-05 00:00:03,956: INFO: common: created directory at: artifacts]
[2026-05-05 00:00:03,957: INFO: common: created directory at: artifacts/prepare_base_model]
58889256/58889256 [==============================] - 3s 0us/step
[2026-05-05 00:00:07,115: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]
[2026-05-05 00:00:07,159: WARNING: optimizer: At this time, the v2.11+ optimizer `tf.keras.optimizers.SGD` runs slowly on M1/M2 Macs, please use the legacy Keras optimizer instead, located at `tf.keras.optimizers.legacy.SGD`.]
[2026-05-05 00:00:07,160: WARNING: __init__: There is a known slowdown when using v2.11+ Keras optimizers on M1/M2 Macs. Falling back to the legacy Keras optimizer, i.

/opt/anaconda3/envs/kidney/lib/python3.8/site-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
